This notebook is dedicated to fixing the anndata object so that it is useful for publishing the automated annotation (of coregulons) on platybrowser

In [1]:
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Define the path to your file (update this path)
FILE_PATH = Path('data/scrnaseq_atlas/atlas_6dpf.h5ad')

# Load the AnnData object
adata = sc.read_h5ad(FILE_PATH)

# Check the object to confirm loading and see available metadata
print(adata)
print(adata.obs.head())

AnnData object with n_obs × n_vars = 31121 × 69614
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mtdna', 'percent.rrna', 'percent.goodgenes', 'curated_high_res_annotations', 'Phase', 'S.Score', 'G2M.Score', 'predicted_group_full', 'stage', 'batch', 'protocol', 'source_material', 'nCount_SCT', 'nFeature_SCT', 'broad.types'
    uns: 'altExperiments', 'version'
    obsm: 'X_pca', 'X_umap'
    layers: 'RNA_counts', 'RNA_data'
                         orig.ident  nCount_RNA  nFeature_RNA  percent.mtdna  \
SN019_AAACCCAAGACAACAT-1      SN019      1447.0         975.0       0.138217   
SN019_AAACCCAGTTCTTAGG-1      SN019      3512.0        2250.0       0.142369   
SN019_AAACGAACATACAGCT-1      SN019      2903.0        1863.0       0.137788   
SN019_AAACGAAGTCTCGGGT-1      SN019      1253.0         938.0       0.718276   
SN019_AAACGCTAGATTGACA-1      SN019      2341.0        1735.0       0.299018   

                          percent.rrna  percent.goodgenes  \
SN019_AAACCCAAGA

In [5]:
# Assuming 'adata' is your loaded AnnData object

broad_types_unique = adata.obs['broad.types'].unique()

print("--- Unique Broad Types (Full List) ---")
# Use list comprehension and join to display the full content
print('\n'.join(map(str, broad_types_unique.tolist())))


--- Unique Broad Types (Full List) ---
Cardiovascular system
Midgut
Motile cilia
Central nervous system
Stem cells
Somatic musculature
Developing neurons
Epidermis
Gland cells
Heme/chitin
Anterior nervous system
Glia
Macrophage
Enteric nervous system
Adult eye


In [14]:
# Assuming 'adata' is your loaded AnnData object

GROUP_COL = 'broad.types'
DETAIL_COL = 'curated_high_res_annotations'

# 1. Group the adata.obs DataFrame and calculate multiple metrics
group_summary = (
    adata.obs.groupby(GROUP_COL)
    .agg(
        total_cells=('nCount_RNA', 'size'),  # Count the number of rows/cells
        unique_annotations=(DETAIL_COL, lambda x: x.unique().tolist()) # Aggregate unique annotations
    )
    .reset_index()
)

print("\n--- Detailed Annotations within Each Broad Grouping (with Cell Count) ---")

# 2. Print the results with the cell count added
for index, row in group_summary.iterrows():
    broad_type = row[GROUP_COL]
    cell_count = row['total_cells']
    annotations = row['unique_annotations']
    
    # Format the cell count for clarity
    count_str = f"{cell_count} cells"
    
    # Use join to display the annotations fully (bypassing the list truncation)
    annotations_str = ', '.join(annotations)
    
    # Print the combined output
    print(f"\n[{broad_type}] ({count_str}):")
    print(f"  {annotations_str}")


--- Detailed Annotations within Each Broad Grouping (with Cell Count) ---

[Central nervous system] (6644 cells):
  clade11sub6subsub10, clade11sub11subsub0-4, clade11sub21, clade11sub35, clade11sub4subsub6, clade11sub4subsub2-5-8, clade11sub16subsub8, clade11sub49, clade11sub11subsub1, clade11sub13subsub6, clade11sub8subsub1-4, clade11sub34, clade11sub18subsub8, clade11sub28subsub2, clade11sub29subsub5, clade11sub23, clade11sub25subsub8, clade11sub4subsub7, clade11sub18subsub6, clade11sub20subsub0-1-2-4-6, clade11sub6subsub7, clade11sub32subsub1, clade11sub18subsub2, clade11sub8subsub8, clade11sub22subsub6, clade11sub11subsub2-3, clade11sub15subsub8, clade11sub27subsub1, nocladesub19, clade11sub37, clade11sub8subsub0, clade11sub15subsub6, clade11sub9subsub8, clade11sub5subsub4, clade11sub42, clade11sub20subsub5, clade11sub32subsub0, clade11sub54subsub0, clade11sub17subsub6, clade11sub44, clade11sub16subsub9, clade11sub14subsub8, clade11sub9subsub4, clade11sub13subsub5, clade11sub16su

C:\Users\Platy\AppData\Local\Temp\ipykernel_11572\3662955861.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(GROUP_COL)


In [15]:
# Assuming 'adata' is your loaded AnnData object

# 1. Define the column and values to remove
COLUMN_TO_FILTER = 'broad.types'
VALUES_TO_REMOVE = ['Macrophage', 'Glia']

# 2. Create a boolean mask: True for all cells that are in the list of values to REMOVE
mask_remove = adata.obs[COLUMN_TO_FILTER].isin(VALUES_TO_REMOVE)

# 3. Apply the INVERSE mask to keep all other cells
# [~mask_remove, :] selects all rows (cells) that are NOT Macrophage or Glia.
adata_filtered = adata[~mask_remove, :].copy()

# Print confirmation and new shape
print(f"Original number of cells: {adata.n_obs}")
cells_removed = adata.n_obs - adata_filtered.n_obs
print(f"Number of Macrophage and Glia cells removed: {cells_removed}")
print(f"New AnnData object shape: {adata_filtered.shape}")

Original number of cells: 31121
Number of Macrophage and Glia cells removed: 870
New AnnData object shape: (30251, 69614)


In [18]:
# Assuming 'adata_filtered' is the AnnData object after removing Macrophage and Glia.

COLUMN_TO_MODIFY = 'broad.types'

# --- Consolidation Mapping ---
# All keys will be replaced by the single value 'Nervous System'
CONSOLIDATION_MAP = {
    'Central nervous system': 'Nervous system',
    'Adult eye':              'Nervous system',
    'Developing neurons':     'Nervous system'
}

# --- Single Rename Mapping ---
SINGLE_RENAME_MAP = {
    'Motile cilia': 'Sensory motile cilia'
}

# 1. Apply Consolidation
adata_filtered.obs[COLUMN_TO_MODIFY] = (
    adata_filtered.obs[COLUMN_TO_MODIFY]
    .replace(CONSOLIDATION_MAP)
)

# 2. Apply Single Rename
adata_filtered.obs[COLUMN_TO_MODIFY] = (
    adata_filtered.obs[COLUMN_TO_MODIFY]
    .replace(SINGLE_RENAME_MAP)
)

print("--- Group Modification Complete ---")
# Display the counts for the new groups
print(adata_filtered.obs[COLUMN_TO_MODIFY].value_counts().head(5))

--- Group Modification Complete ---
broad.types
Nervous System          8493
Somatic musculature     5117
Epidermis               4995
Sensory Motile Cilia    2569
Midgut                  2413
Name: count, dtype: int64


In [19]:
OUTPUT_FILENAME = 'data/scrnaseq_atlas/updated_6dpf_atlas.h5ad'

# Save the AnnData object to the specified file path
adata_filtered.write(OUTPUT_FILENAME)

print(f"✅ Successfully saved the modified AnnData object to: {OUTPUT_FILENAME}")
print(f"New object contains all cells EXCEPT Macrophage and Glia.")


✅ Successfully saved the modified AnnData object to: data/scrnaseq_atlas/updated_6dpf_atlas.h5ad
New object contains all cells EXCEPT Macrophage and Glia.
